# Reproducibility

Record both the random seed and the computational environment when saving an
analysis. A seed controls random draws; it does not by itself guarantee identical
floating-point results across environments.

In [1]:
import importlib.metadata
import platform

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import tensorflow_probability.substrates.jax.bijectors as tfb
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.goose as gs
import liesel.model as lsl

## PRNG seeding

This example uses the regression model from {doc}`01c-transform`, including its
kernel assignments and jitter settings. It starts from that model's initial
state, independently of earlier notebook sessions.

In [2]:
# Python cell fragment: imports are visible in each guide.
# Simulate regression observations.
rng = np.random.default_rng(42)
n = 500
true_beta = np.array([1.0, 2.0])
true_sigma = 1.0
x = rng.uniform(size=n)
X_mat = np.column_stack([np.ones(n), x])
y_vec = X_mat @ true_beta + rng.normal(scale=true_sigma, size=n)

# Define the priors.
beta = lsl.Var.new_param(
    jnp.zeros(2),
    dist=lsl.Dist(tfd.Normal, 0.0, 5.0),
    name="beta",
)
sigma_sq = lsl.Var.new_param(
    1.0,
    dist=lsl.Dist(tfd.InverseGamma, concentration=3.0, scale=2.0),
    name="sigma_sq",
)

# Connect the parameters to the observations.
sigma = lsl.Var.new_calc(jnp.sqrt, sigma_sq, name="sigma")
X = lsl.Var.new_obs(X_mat, name="X")
mu = lsl.Var.new_calc(jnp.dot, X, beta, name="mu")
y = lsl.Var.new_obs(
    y_vec,
    dist=lsl.Dist(tfd.Normal, mu, sigma),
    name="y",
)

sigma_sq.biject(tfb.Exp(), name="log_sigma_sq")
log_sigma_sq = sigma_sq.bijected_var

joint = gs.MCMCSpec(
    gs.NUTSKernel,
    kernel_group="regression",
    jitter_dist=tfd.Normal(0.0, 0.2),
)
beta.inference = joint
log_sigma_sq.inference = joint
model = lsl.Model(y)


In [3]:
results = gs.LieselMCMC(model).run_for_epochs(
    seed=1,
    num_chains=4,
    adaptation=500,
    posterior=500,
    show_progress=False,
)
samples = results.get_posterior_samples()

liesel.goose.engine - INFO - Initializing kernels...


liesel.goose.engine - INFO - Done


liesel.goose.engine - INFO - Finished warmup


In [4]:
{name: value.shape for name, value in samples.items()}

{'beta': (4, 500, 2), 'log_sigma_sq': (4, 500)}

The seed configures Goose's random-key stream, including chain initialization.
Keep data, initial values, jitter, kernel ordering, and iteration counts with
it. The lower-level {class}`~liesel.goose.EngineBuilder` also takes a seed.

[JAX uses explicit random keys](https://docs.jax.dev/en/latest/random-numbers.html).
When simulating data yourself, split a key for separate draws. Reusing a key
repeats its random stream; it does not produce an independent replicate.

In [5]:
x_key, noise_key = jax.random.split(jax.random.key(42))
x = jax.random.normal(x_key, (3,))
noise = jax.random.normal(noise_key, (3,))

In [6]:
{"x": x, "noise": noise}

{'x': Array([ 0.07592554, -0.48634264,  1.2903206 ], dtype=float32),
 'noise': Array([ 0.60576403,  0.7990441 , -0.908927  ], dtype=float32)}

<a id="practical-checklist"></a>

## Record the environment

Capture versions and numerical settings from the environment actually running
the analysis. Keep the source code and input data alongside this information.

In [7]:
packages = ["liesel", "jax", "jaxlib", "tfp-nightly", "blackjax"]
environment = {
    "Python": platform.python_version(),
    "Platform": platform.platform(),
    "Backend": jax.default_backend(),
    "64-bit enabled": jax.config.jax_enable_x64,
    "PRNG implementation": jax.config.jax_default_prng_impl,
    **{name: importlib.metadata.version(name) for name in packages},
}

In [8]:
pd.Series(environment, name="Value").to_frame()

,Value
Python,3.13.15
Platform,Linux-6.17.0-1022-azure-x86_64-with-glibc2.39
Backend,cpu
64-bit enabled,False
PRNG implementation,threefry2x32
liesel,0.6.0.dev0+optim.1
jax,0.10.1
jaxlib,0.10.1
tfp-nightly,0.26.0.dev20260528
blackjax,1.5


Also record the exact source revision if using development code, package lock
file, model and sampler settings, and data preparation. Save posterior draws
and diagnostics for a published analysis; rerunning code is not a substitute
for preserving the results being reported.

<a id="gpu-non-determinism"></a>
<a id="non-reproducibility-across-systems"></a>

## Understand the limits

Changing numerical libraries, hardware, compiler settings, or evaluation order
can change floating-point calculations. In MCMC, small differences can affect
adaptation or acceptance decisions and lead to different trajectories. CPU
execution is not a promise of bitwise equality across systems.

The [Stan reproducibility discussion](https://mc-stan.org/docs/reference-manual/reproducibility.html)
also distinguishes repeating a program in a fixed computational environment
from statistical reproducibility. When comparing runs in different environments,
assess posterior quantities relative to their Monte Carlo uncertainty and check
{doc}`../../goose-diagnostics`; do not require identical individual draws.

<a id="see-also"></a>

For this documentation, the repository lockfile and
{download}`build instructions <../../../README.md>`
specify the tested environment. See the
[JAX random configuration reference](https://docs.jax.dev/en/latest/config_options.html)
for PRNG options.